# Suite EMB — Embedded objects and union values

A property whose schema is an `OfObject` holds an embedded object: a read-only record with no identity, written nested in
its owner's snapshot. A property whose schema is an `OfUnion` holds a value of one of its branches, recorded with the
branch it is written as (`{"$branch": index, "$value": value}` in snapshots). Validation checks union values against
their branches' predicates when given an evaluator; these cases use a stub for predicates written as `("has", name)`
and `("type", name)`.

In [ ]:
import { Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { AttributeError, DecodeError, NotImplementedError, ValueError } from "@mbse/schemas/Framework/Errors";
import type { OfObject, Visitable } from "@mbse/schemas/Framework/Visitors";
import { assert, equal, Fake, map, raises, same, set_property, stub_evaluator, text } from "./support.js";

const Phone = new S.OfObject.Builder().properties(text("number"), text("label")).create();
const Email = new S.OfObject.Builder().properties(text("address")).create();
const Reach = new S.OfUnion.Builder().branches((b) => b.of(Phone).when(["has", "number"]),
  (b) => b.of(Email).when(["has", "address"])).create();
const Id = new S.OfUnion.Builder().branches((b) => b.of((t) => t.as_native(BigInt)).when(["type", "int"]),
  (b) => b.of((t) => t.as_native(String)).when(["type", "str"])).create();
const Contact = new S.OfObject.Builder().properties(
  text("name"), (p) => p.name("home").of(Phone), (p) => p.name("reach").of(Reach), (p) => p.name("id").of(Id)).create();
Proxies.register("Contact", Contact);
const B = Proxies.Builders;

/** A `Visitable` whose `accept` runs `write(visitor)`: data that neither proxies nor fakes produce. */
class Written implements Visitable {
  constructor(private readonly schemaName: string, private readonly write: (visitor: any) => void) {}

  identity(): unknown {
    return this;
  }

  schema_name(): string {
    return this.schemaName;
  }

  accept(visitor: OfObject): void {
    this.write(visitor);
  }
}

## EMB-01 · An embedded object's schema has no adjacencies, however it is held

In [ ]:
assert(Contact.validate().length === 0 && Reach.validate().length === 0 && Id.validate().length === 0);
let Owner = new S.OfObject.Builder().properties(text("x")).create();
const Links = new S.OfRelation.Builder().links("a", "b").create();
Owner = new S.OfObject.Builder(Owner).relations((r) => r.name("links").of(Links).me("a")).update();
const Either = new S.OfUnion.Builder().branches((b) => b.of(Owner).when(["has", "x"]),
  (b) => b.of(Phone).when(["has", "number"])).create();
const Both = new S.OfIntersection.Builder().of(Owner, Phone).create();
const Holder = new S.OfObject.Builder().properties((p) => p.name("owner").of(Owner), (p) => p.name("either").of(Either),
  (p) => p.name("both").of(Both)).create();
assert(same(Holder.validate(), [
  "property 'owner': an embedded object cannot have adjacencies",
  "property 'either': an embedded object cannot have adjacencies",
  "property 'both': an embedded object cannot have adjacencies",
]));
const Pair = new S.OfRelation.Builder().links("a", "b").properties((p) => p.name("inner").of(Owner)).create();
assert(same(Pair.validate(), ["property 'inner': an embedded object cannot have adjacencies"]));

## EMB-02 · Embedded objects: set with a Spec or a record, read, updated in part, cleared

In [ ]:
const alice = B.Contact().name("Alice").home((r: any) => r.number("+44 1").label("home")).create();
assert(alice.home.number === "+44 1" && alice.home.label === "home");
raises(AttributeError, () => B.Contact().home((r: any) => r.number("1")).create().home.label, "property 'label' is not set");
raises(AttributeError, () => { alice.home.number = "x"; }, "embedded objects are read-only; use a builder");
raises(AttributeError, () => alice.home.nope, "nope");
assert(B.Contact(alice).home((r: any) => r.label("work")).update() === alice); // starts from the value already set
assert(alice.home.number === "+44 1" && alice.home.label === "work");
const bob = B.Contact().name("Bob").home(alice.home).create(); // a record is a value, copied into the new owner
assert(bob.home.number === "+44 1" && bob.home.label === "work");
B.Contact(bob).clear("home").update();
raises(AttributeError, () => bob.home, "property 'home' is not set");
assert(alice.home.label === "work");
raises(TypeError, () => B.Contact().home("+44"), "property 'home' takes an embedded object or a Spec, got str");
raises(AttributeError, () => B.Contact().home((r: any) => r.nope(1n)), "'nope' is not a property of the embedded object");

## EMB-03 · Union values: the value's kind chooses the branch, or `of` and `select` do

In [ ]:
const carol = B.Contact().reach((u: any) => u.of(Email, (r: any) => r.address("c@x"))).id(7n).create();
assert(carol.reach.address === "c@x" && carol.id === 7n);
const dan = B.Contact().reach(carol.reach).id("D-1").create(); // the record's schema, the native's type
assert(dan.reach.address === "c@x" && dan.id === "D-1");
raises(TypeError, () => B.Contact().id(1.5), "no branch of the union holds a float");
raises(TypeError, () => B.Contact().id(carol.reach), "no branch of the union holds an embedded object");
raises(ValueError, () => B.Contact().reach((u: any) => u.of(Contact, (r: any) => r)), "no branch of the union has that schema");
raises(ValueError, () => B.Contact().reach((u: any) => u.select(5)), "the union has no branch 5");
raises(ValueError, () => B.Contact().reach((u: any) => u.value(() => undefined)), "select a branch before writing the union's value");
const seen: unknown[] = [];
B.Contact(carol).property("reach", (p) => p.value((a) => a.as_union((u) => seen.push(u.branch()))));
assert(same(seen, [1]));
raises(AttributeError, () => B.Contact().property("reach", (p) => p.value((a) => a.as_union((u) => u.branch()))),
  "property 'reach' is not set");
B.Contact(carol).id((u: any) => u.value((v: any) => v.as_native((n: any) => n.clear()))).update(); // clearing the value
raises(AttributeError, () => carol.id, "property 'id' is not set");
B.Contact(carol).reach((u: any) => u.select(0).value((v: any) => v.as_object((r: any) => r.number("9")))).update();
assert(carol.reach.number === "9"); // another branch: its value starts empty
raises(TypeError, () => B.Contact().property("name", (p) => p.value((a) => a.as_object(() => undefined))),
  "property 'name' does not hold an object");
raises(TypeError, () => B.Contact().property("name", (p) => p.value((a) => a.as_union(() => undefined))),
  "property 'name' does not hold a union");

## EMB-04 · Snapshots: embedded objects nested, union values with their branch; strict decoding

In [ ]:
const eve = B.Contact().name("Eve").home((r: any) => r.number("1"))
  .reach((u: any) => u.of(Phone, (r: any) => r.number("2").label("m"))).id("E").create();
const expected = JSON.ToJSON(Contact, eve);
assert(expected === '{"root": "s0", "objects": {"s0": {"name": "Eve", "home": {"number": "1"}, "reach": {"$branch": 0, '
  + '"$value": {"number": "2", "label": "m"}}, "id": {"$branch": 1, "$value": "E"}}}}');
assert(JSON.ToJSON(Contact, Plain.FromPlain(B)(Contact, Plain.ToPlain(Contact, eve))) === expected);
assert(JSON.ToJSON(Contact, JSON.FromJSON(B)(Contact, JSON.ToJSON(Contact, eve))) === expected);
assert(JSON.ToJSON(Contact, YAML.FromYAML(B)(Contact, YAML.ToYAML(Contact, eve))) === expected);

const FromPlain = Plain.FromPlain(B);
for (const [s0, message] of [
  [{ home: 3n }, "$.objects.s0.home: an embedded object must be a mapping, got int"],
  [{ home: { x: "1" } }, "$.objects.s0.home.x: the embedded object has no property 'x'"],
  [{ home: { number: 1n } }, "$.objects.s0.home.number: expected str, got int"],
  [{ reach: { number: "1" } }, "$.objects.s0.reach: a union value is {'$branch': index, '$value': value}"],
  [{ id: { $branch: 2n, $value: 1n } }, "$.objects.s0.id.$branch: the union has no branch 2"],
  [{ id: { $branch: true, $value: 1n } }, "$.objects.s0.id.$branch: the union has no branch True"],
  [{ id: { $branch: 0n, $value: "x" } }, "$.objects.s0.id.$value: expected int, got str"],
] as const) {
  raises(DecodeError, () => FromPlain(Contact, map({ root: "s0", objects: { s0 } })), message);
}
raises(TypeError, () => Plain.ToPlain(Contact, new Fake("Contact", { home: "x" })), "property 'home' is not native");

let Box = new S.OfObject.Builder().properties(text("x")).create();
const Slot = new S.OfRelation.Builder().links("box", "item").properties((p) => p.name("tag").of(Phone)).create();
Box = new S.OfObject.Builder(Box).relations((r) => r.name("items").of(Slot).me("box"),
  (r) => r.name("boxes").of(Slot).me("item")).update();
Proxies.register("Box", Box);
Proxies.register("Slot", Slot);
raises(NotImplementedError, () => FromPlain(Box, map({ root: "s0", objects: { s0: { items: [
  { item: { $ref: "s0", $schema: "Box" }, tag: "x" }] } } })), "entry property 'tag': entry properties must be native");

## EMB-05 · Validation: embedded values recursively, and union branches with an evaluator

In [ ]:
const validate = Validators.Validate(B), checked = Validators.Validate(B, stub_evaluator);
assert(validate(Contact, eve).length === 0 && checked(Contact, eve).length === 0);
const odd = B.Contact().reach((u: any) => u.of(Phone, (r: any) => r.label("no number"))).create();
assert(validate(Contact, odd).length === 0); // without an evaluator, only kinds and types are checked
assert(same(checked(Contact, odd), ["Contact#0.reach: no branch's predicate holds for the value"]));
const Twice = new S.OfUnion.Builder().branches((b) => b.of((t) => t.as_native(String)).when(["type", "str"]),
  (b) => b.of((t) => t.as_native(String)).when(["type", "str"])).create();
const Note = new S.OfObject.Builder().properties((p) => p.name("text").of(Twice)).create();
Proxies.register("Note", Note);
const note = B.Note().text((u: any) => u.select(1).value((v: any) => v.as_native((n: any) => n.set("t")))).create();
assert(same(checked(Note, note), ["Note#0.text: written as branch 1, but the first branch whose predicate holds is 0"]));
assert(same(validate(Contact, new Fake("Contact", { home: "x", reach: 3n })), [
  "Contact#0.home: expected an embedded object, got str", "Contact#0.reach: expected a union value, got int"]));
assert(same(validate(Contact, B.Contact().home((r: any) => r.number(5n)).create()),
  ["Contact#0.home.number: expected str, got int"]));
const stray = new Written("Contact", (v) => v.property("home", (p: any) => p.value((a: any) => a.as_object(
  (o: any) => set_property(o, "fax", "1")))));
assert(same(validate(Contact, stray), ["Contact#0.home.fax: not a property of the embedded object"]));
const nine = new Written("Contact", (v) => v.property("id", (p: any) => p.value((a: any) => a.as_union(
  (u: any) => u.select(9).value((w: any) => w.as_native((n: any) => n.set(1n)))))));
assert(same(validate(Contact, nine), ["Contact#0.id: the union has no branch 9"]));
assert(same(validate(Contact, new Written("Contact", (v) => v.property("id", (p: any) => p.value((a: any) => a.as_object(
  () => undefined))))), ["Contact#0.id: expected a union value, got an embedded object"]));
const Joint = new S.OfIntersection.Builder().of(Phone, Email).create();
const Mixed = new S.OfObject.Builder().properties((p) => p.name("both").of(Joint)).create();
Proxies.register("Mixed", Mixed);
assert(same(validate(Mixed, new Fake("Mixed", { both: 1n })), ["Mixed#0.both: intersection values cannot be validated yet"]));
raises(NotImplementedError, () => FromPlain(Mixed, map({ root: "s0", objects: { s0: { both: 1n } } })),
  "intersection values are not supported by Plain yet");
assert(same(validate(Contact, new Written("Contact", (v) => v.property("name", (p: any) => p.value((a: any) => a.as_union(
  (u: any) => u.select(0).value((w: any) => w.as_native((n: any) => n.set(1n)))))))),
  ["Contact#0.name: expected str, got a union value"]));
const box = new Fake("Box");
box.adjacencies.set("items", [{ item: box, tag: "x" }]);
assert(same(validate(Box, box), ["Box#0.items[0].tag: entry properties must be native"]));

const values = Validators.properties_of(eve);
assert(values.get("id") === "E"); // a union value reads as the value it holds
assert(equal(Validators.properties_of(values.get("home") as Visitable), map({ number: "1" })));
assert(equal(Validators.properties_of(values.get("reach") as Visitable), map({ number: "2", label: "m" })));
const Tagged = new S.OfObject.Builder().properties((p) => p.name("id").of(Id)).create();
const Sleeve = new S.OfObject.Builder().properties((p) => p.name("tagged").of(Tagged)).create();
Proxies.register("Sleeve", Sleeve);
const sleeve = B.Sleeve().tagged((r: any) => r.id(5n)).create();
assert(equal(Validators.properties_of(Validators.properties_of(sleeve).get("tagged") as Visitable), map({ id: 5n })));

## EMB-06 · Comparison: embedded objects by their properties, union values by branch and value

In [ ]:
const record = (value: Visitable) => new C.OfObject(Contact, value);

assert(record(eve).compare(record(B.Contact(eve).clone())) === 0);
assert(record(eve).compare(record(B.Contact(eve).home((r: any) => r.label("x")).clone())) === null);
assert(record(eve).compare(record(B.Contact(eve).id(3n).clone())) === null); // another branch
assert(record(eve).compare(record(B.Contact(eve).id("F").clone())) === null);
assert(record(B.Contact().create()).compare(record(B.Contact().create())) === 0);
const union = new C.OfUnion(Id);
raises(ValueError, () => union.branch(), "no branch is selected");
raises(ValueError, () => union.select(2), "the union has no branch 2");
raises(ValueError, () => union.select(1.5), "the union has no branch 1.5");
assert(union.compare(new C.OfUnion(Id).select(0).value((v) => v.as_native((n) => n.set(1n)))) === null); // one is absent
union.select(0).value((v) => v.as_native((n) => n.set(1n)));
assert(union.select(0).branch() === 0 && union.compare(new C.OfUnion(Reach)) === null); // reselecting keeps the value
const other = new C.OfUnion(Id).select(0).value((v) => v.as_native((n) => n.set(1n)));
assert(union.compare(other) === 0 && other.compare(new C.OfUnion(Id)) === null);
assert(new C.OfUnion(Id).compare(new C.OfUnion(Id)) === 0); // both are absent
const emptied = new C.OfAny(Id).as_union((u) => u.select(0).value((v) => v.as_native((n) => n.set(1n).clear())));
assert(emptied.compare(new C.OfAny(Id)) === 0); // a union whose value was cleared is absent
const word = new C.OfAny(new S.OfNative.Data(String)).as_native((n) => n.set("b"));
assert(new C.OfAny(Id).as_union((u) => u.select(1).value((v) => v.as_native((n) => n.set("b")))).compare(word) === null);
const twice = new C.OfAny(Id);
twice.as_union((u) => u.select(1).value((v) => v.as_native((n) => n.set("a"))));
twice.as_union((u) => u.value((v) => v.as_native((n) => n.set("b"))));
assert(twice.compare(new C.OfAny(Id).as_union((u) => u.select(1).value((v) => v.as_native((n) => n.set("b"))))) === 0);

## EMB-07 · The new visitor classes read back through their protocols

In [ ]:
// Proxies: an embedded object's builder, read through Visitors.OfObject.
const heard: unknown[] = [];
B.Contact(alice).property("home", (p) => p.value((a) => a.as_object((r) => {
  r.properties((q) => heard.push(q.name()));
  heard.push(r.has("label"));
  r.clear("label");
  heard.push(r.has("label"));
  r.adjacencies((x) => heard.push(x));
})));
assert(same(heard, ["number", "label", true, false]));
raises(AttributeError, () => B.Contact().home((r: any) => r.adjacency("x", () => undefined)),
  "an embedded object has no adjacencies, got 'x'");
raises(AttributeError, () => B.Contact().home((r: any) => r.property("nope", () => undefined)),
  "'nope' is not a property of the embedded object");
B.Contact().home((r: any) => heard.push(r.then)); // the runtime's own probes are not properties
B.Contact().home((r: any) => heard.push(r[Symbol.toStringTag]));
assert(heard[heard.length - 1] === undefined && heard[heard.length - 2] === undefined);
assert(alice.home[Symbol.iterator] === undefined);

// Plain: writers for embedded objects and unions, and their errors.
const out = new Map() as Plain.PlainMap;
const writer = new Plain._ObjectWriter(out, Contact, () => new Map());
writer.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "number", "1"))));
writer.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "label", "h"))));
assert(equal(out, map({ home: { number: "1", label: "h" } }))); // writing again adds to the same mapping
raises(TypeError, () => writer.property("name", (p) => p.value((a) => a.as_object(() => undefined))),
  "property 'name' is not an object");
raises(TypeError, () => writer.property("name", (p) => p.value((a) => a.as_union(() => undefined))),
  "property 'name' is not a union");
writer.property("id", (p) => p.value((a) => a.as_union((u) => {
  heard.push(u.select(1).branch());
  u.value((v) => v.as_native((n) => n.set("I")));
})));
assert(equal(out.get("id"), map({ $branch: 1n, $value: "I" })));
writer.property("id", (p) => p.value((a) => a.as_union((u) => u.select(1).value(() => undefined))));
assert(!out.has("id")); // nothing written: no value
raises(ValueError, () => writer.property("id", (p) => p.value((a) => a.as_union((u) => u.branch()))), "no branch is selected");
raises(ValueError, () => writer.property("id", (p) => p.value((a) => a.as_union((u) => u.select(3)))), "the union has no branch 3");

// Validators: the recorder for union values.
const recorder = new Validators._ObjectRecord();
recorder.property("id", (p) => p.value((a) => a.as_union((u) => u.select(0).value(
  (v) => v.as_native((n) => n.set(1n))))));
recorder.property("id", (p) => p.value((a) => a.as_union((u) => heard.push(u.branch()))));
recorder.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "number", "1"))));
recorder.property("home", (p) => p.value((a) => a.as_object((o) => set_property(o, "label", "h"))));
assert(equal(Validators.properties_of(Validators.properties_of(recorder).get("home") as Visitable),
  map({ number: "1", label: "h" })));
assert(heard[heard.length - 1] === 0);
raises(ValueError, () => new Validators._UnionRecord().branch(), "no branch is selected");